# YAMNet Exploration

Goal:
Understand how pretrained audio models interpret sound.

Questions:

- What does YAMNet output?
- What are embeddings?
- How fast is inference?
- Does it recognize our sounds?

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
from pathlib import Path
import librosa
import pandas as pd

In [ ]:
model = hub.load(
    "https://tfhub.dev/google/yamnet/1"
)

model

In [ ]:
ROOT = Path("../data/raw/esc50")

AUDIO = ROOT / "audio"

sample = AUDIO / "1-22694-A-20.wav"

In [ ]:
waveform, sr = librosa.load(
    sample,
    sr=16000
)

waveform = waveform.astype(
    np.float32
)

print(sr)
print(waveform.shape)

In [ ]:
scores, embeddings, spectrogram = model(
    waveform
)

In [ ]:
print(scores.shape)

print(embeddings.shape)

print(spectrogram.shape)

In [ ]:
scores.numpy()[:3]

In [ ]:
embeddings.numpy()[:2]

Observation

- Scores: The model broke our 5-second audio clip down into 10 distinct temporal windows (frames). For each window, it outputs a vector of 521 probability scores, representing YAMNet's confidence across its 521 supported sound classes.
- Embeddings: The timeline matches the 10 windows seen in the scores. Each frame is represented by a highly dense 1,024-dimensional feature vector. These embeddings capture the core acoustic signature of the sound and are ideal for downstream transfer learning.
- Spectrogram: This represents YAMNet's internal preprocessing step. It computed a Log-Mel Spectrogram across the full audio clip containing 528 time slices and compressed the frequency information into 64 mel bins.

In [ ]:
class_map = pd.read_csv(
    "https://raw.githubusercontent.com/tensorflow/models/master/research/audioset/yamnet/yamnet_class_map.csv"
)

In [ ]:
prediction = scores.numpy().mean(
    axis=0
)

In [ ]:
top = prediction.argsort()[-10:][::-1]

class_map.iloc[top]

Alarm:
strong

Knock:
good

Baby:
weak

In [ ]:
import time

start = time.time()

model(
    waveform
)

end = time.time()

print(
    end-start
)